# 05.09_get_KEGG_R

构建 KEGG KO 与通路映射。

- 当前文件：`analysis/05_genome_analysis/05.09_get_KEGG_R.ipynb`
- 原始来源：`Codes/05.09_R_get_KEGG.ipynb`（旧编号仅用于溯源）。
- 运行内核：**R**。
- 导入依赖：`RCurl`, `dplyr`, `jsonlite`, `purrr`, `stringr`。
- 当前编号与流程见 `docs/workflow.md`、`docs/code_index.md`。
- 仅更新整理版导读；原分析单元格、参数和顺序保持不变。原始 cell 索引在本文件中加 1。


## KEGG注释

In [ ]:
# 需要下载 json 文件 (这是是经常更新的)
# https://www.genome.jp/kegg-bin/get_htext?ko00001
# 代码来自：http://www.genek.tv/course/225/task/4861/show
library (jsonlite)
library (purrr)
library (RCurl)
library (dplyr)
library (stringr)
kegg <- function (json = "ko00001.json") {
    pathway2name <- tibble (Pathway = character (), Name = character ())
    ko2pathway <- tibble (Ko = character (), Pathway = character ())
    
    kegg <- fromJSON (json)
    
    for (a in seq_along (kegg [["children"]][["children"]])) {
      A <- kegg [["children"]][["name"]][[a]]
      
      for (b in seq_along (kegg [["children"]][["children"]][[a]][["children"]])) {
        B <- kegg [["children"]][["children"]][[a]][["name"]][[b]] 
        
        for (c in seq_along (kegg [["children"]][["children"]][[a]][["children"]][[b]][["children"]])) {
          pathway_info <- kegg [["children"]][["children"]][[a]][["children"]][[b]][["name"]][[c]]
          
          # pathway_id <- str_match (pathway_info, "ko [0-9]{5}")[1]
          pathway_id <- str_match (pathway_info, "ko[0-9]{5}")[1]
          pathway_name <- str_replace (pathway_info, "\\[PATH:ko [0-9]{5}\\]", "") %>% str_replace ("[0-9]{5} ","")
          pathway2name <- rbind (pathway2name, tibble (Pathway = pathway_id, Name = pathway_name))
          
          kos_info <- kegg [["children"]][["children"]][[a]][["children"]][[b]][["children"]][[c]][["name"]]
          
          kos <- str_match (kos_info, "K [0-9]*")[,1]
          
          ko2pathway <- rbind (ko2pathway, tibble (Ko = kos, Pathway = rep (pathway_id, length (kos))))
        }
      }
    }
    colnames (ko2pathway) <- c ("KO","Pathway")
    save (pathway2name, ko2pathway, file = "/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/GO_and_KEGG/kegg_info.RData")
    write.table (pathway2name,"/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/GO_and_KEGG/KEGG.library",sep="\t",row.names = F)
  }
  
  
kegg (json = "/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/GO_and_KEGG/ko00001.json")